<a href="https://colab.research.google.com/github/kiriakosgp/papadopoulos_av_analysis/blob/main/bertopic_comments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install BERTopic UMAP-learn hdbscan

In [ ]:
import pandas as pd
import numpy as np
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
import os
import re

In [ ]:
def build_topic_model(min_cluster_size=15):
    umap_model = UMAP(
        n_neighbors=10,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=5,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True
    )

    vectorizer_model = CountVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=5
    )

    return BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        nr_topics="auto",
        calculate_probabilities=True,
        verbose=True
    )

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\b\d+\b", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [ ]:
COMMENT_EMB_DIR = "/content/drive/MyDrive/comments_agd_embeddings"
DATASET_DIR = "/content/drive/MyDrive/dataset"

documents = []
embeddings = []
video_ids = []

for emb_fname in os.listdir(COMMENT_EMB_DIR):
    if not emb_fname.endswith(".npy"):
        continue

    video_id = emb_fname.replace(".npy", "")
    info_path = os.path.join(DATASET_DIR, video_id, "info.json")

    if not os.path.exists(info_path):
        continue

    info = pd.read_json(info_path)
    comments = info.get("comments", [])

    if len(comments) == 0:
        continue

    text = " ".join(comments).strip()
    if len(text) < 20:
        continue

    text = preprocess_text(text)

    emb_path = os.path.join(COMMENT_EMB_DIR, emb_fname)
    video_emb = np.load(emb_path)

    documents.append(text)
    embeddings.append(video_emb)
    video_ids.append(video_id)


embeddings = np.vstack(embeddings) if embeddings else np.array([])


In [ ]:
print(f"Embedding dim: {embeddings.shape[1]}")
print(f"Docs: {len(documents)}")

In [ ]:
topic_model = build_topic_model(min_cluster_size=15)
topics, probs = topic_model.fit_transform(documents, embeddings)

In [ ]:
df = pd.DataFrame({
    "video_id": video_ids,
    "document": documents,
    "topic": topics,
    "topic_prob": probs.max(axis=1)
})

df.to_csv("comments_topics_test.csv", index=False)

In [ ]:
topic_info = topic_model.get_topic_info()
print(topic_info.head(20))

In [ ]:
for topic_id in topic_info["Topic"]:
    if topic_id == -1:  # skip outliers
        continue
    words = topic_model.get_topic(topic_id)
    print(f"Topic {topic_id}: {', '.join([w for w,_ in words[:10]])}")

In [ ]:
topic_id = 0
print("Top 3 representative transcripts:")
for doc in topic_model.get_representative_docs(topic_id)[:3]:
    print("-", doc[:200], "...")

In [ ]:
fig1 = topic_model.visualize_topics()
fig1.show()

fig2 = topic_model.visualize_barchart(top_n_topics=15)
fig2.show()

fig3 = topic_model.visualize_hierarchy()
fig3.show()

In [ ]:
topic_info[["Topic", "Count"]].sort_values("Count", ascending=False)


In [ ]:
outlier_rate = (df["topic"] == -1).mean()
print(f"Outlier proportion: {outlier_rate*100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

plt.hist(df["topic_prob"], bins=30)
plt.xlabel("Topic assignment probability")
plt.ylabel("Number of transcripts")
plt.title("Topic Probability Distribution")
plt.show()


In [ ]:
sentiment_labels = "/content/drive/MyDrive/comments_sentiment_0shot.csv"
df_sentiment = pd.read_csv(sentiment_labels)
print(df_sentiment.head())
print(df_sentiment.info())

In [ ]:
label_mapping = {"negative": -1, "neutral": 0, "positive": 1}
df_sentiment["sentiment_numeric"] = df_sentiment["label"].map(label_mapping)

In [ ]:
df_sentiment_topics = df.merge(df_sentiment, on="video_id", how="left")

print(df_sentiment_topics.head())

In [ ]:
# Example mapping
topic_labels = {
    0: "US government",
    1: "I/P geo-politics",
    2: "UK internal politics",
    3: "R/U war",
    4: "World news",
    5: "Foreign news",
    6: "India-Pakistan news",
    7: "US relations to China/Canada",
    8: "General news broadcasts",
    9: "Local people stories",
    10: "Slang discussion",
    11: "Discussion on politicians",
    12: "Europe-UK"
}

df["topic_label"] = df["topic"].map(topic_labels)
df_sentiment_topics = df.merge(df_sentiment, on="video_id", how="left")

print(df_sentiment_topics.columns)

In [ ]:
df_sentiment_topics.to_csv("comments_topics_sentiment.csv", index=False)

In [ ]:
df_sentiment_topics.groupby("topic_label")["score"].mean().sort_values(ascending=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(14,6))
sns.countplot(
    data=df_sentiment_topics,
    x="topic_label",
    hue="label",
    palette={"negative":"red", "neutral":"gray", "positive":"green"}
)
plt.xticks(rotation=45, ha='right')
plt.ylabel("Number of transcripts")
plt.title("Sentiment distribution per topic")
plt.legend(title="Sentiment")
plt.show()

In [ ]:
avg_sentiment = df_sentiment_topics.groupby("topic_label")["sentiment_numeric"].mean().sort_values()

plt.figure(figsize=(14,6))
avg_sentiment.plot(kind="bar", color="skyblue")
plt.ylabel("Average sentiment (-1 negative, 0 neutral, 1 positive)")
plt.title("Average sentiment per topic")
plt.xticks(rotation=45, ha='right')
plt.show()

In [ ]:
!pip install gensim

In [ ]:
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary

def compute_coherence(topic_model, documents, coherence_type="c_v", top_n_words=10):

    tokenized_docs = [doc.split() for doc in documents]

    dictionary = Dictionary(tokenized_docs)

    topics = []

    for topic_id in topic_model.get_topics().keys():
        if topic_id == -1:
            continue

        topic_words = topic_model.get_topic(topic_id)

        if topic_words is None or len(topic_words) == 0:
            continue

        words = [word for word, _ in topic_words[:top_n_words]]

        if len(words) == 0:
            continue


        topics.append(words)

    if len(topics) == 0:
        raise ValueError("No valid topics found for coherence calculation.")

    coherence_model = CoherenceModel(
        topics=topics,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence=coherence_type
    )

    return coherence_model.get_coherence()

In [ ]:
coherence_transcript = compute_coherence(
    topic_model,
    documents,
    coherence_type="c_v"
)

print("Transcript Model Coherence (c_v):", coherence_transcript)

In [ ]:
def topic_diversity(topic_model, top_n_words=10):
    topics = []
    for topic_id in topic_model.get_topics().keys():
        if topic_id == -1:
            continue
        words = [word for word, _ in topic_model.get_topic(topic_id)[:top_n_words]]
        topics.append(words)

    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)

    diversity = len(unique_words) / len(all_words)
    return diversity

In [ ]:
print("Transcript Topic Diversity:", topic_diversity(topic_model))

In [ ]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

sil_score = silhouette_score(
    embeddings[mask],
    np.array(topics)[mask]
)

print("Silhouette Score:", sil_score)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import kruskal

transcript_topics = pd.read_csv("/content/drive/MyDrive/transcript_topics.csv")
comment_sentiment = pd.read_csv("/content/drive/MyDrive/comments_sentiment_0shot.csv")


sentiment_map = {
    "negative": -1,
    "neutral": 0,
    "positive": 1
}
comment_sentiment["sentiment_numeric"] = comment_sentiment["label"].map(sentiment_map)

df = transcript_topics.merge(
    comment_sentiment,
    on="video_id",
    how="inner"
)

print(f"Aligned videos: {len(df)}")


topic_summary = (
    df.groupby("topic")["sentiment_numeric"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

print(topic_summary)

groups = [
    g["sentiment_numeric"].values
    for _, g in df.groupby("topic")
]

stat, p = kruskal(*groups)
print(f"Kruskal-Wallis H={stat:.3f}, p={p:.4g}")


plt.figure(figsize=(10, 5))
sns.boxplot(
    data=df,
    x="topic",
    y="sentiment_numeric"
)
plt.axhline(0, linestyle="--", color="gray")
plt.title("Comment Sentiment by Transcript Topic")
plt.xlabel("Transcript Topic")
plt.ylabel("Comment Sentiment")
plt.tight_layout()
plt.show()


In [ ]:
comment_topic_dists, _ = topic_model.approximate_distribution(documents)

comment_topic_df = pd.DataFrame(
    comment_topic_dists,
    columns=[f"t{i}" for i in range(comment_topic_dists.shape[1])]
)

comment_topic_df.insert(0, "video_id", video_ids)


In [ ]:
comment_topic_df.to_csv("comment_topic_dists.csv", index=False)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine
from scipy.stats import spearmanr


transcript_topics = pd.read_csv("/content/drive/MyDrive/topic_distributions.csv")
comment_topics = pd.read_csv("/content/drive/MyDrive/comment_topic_dists.csv")
comment_sentiment = pd.read_csv("/content/drive/MyDrive/comments_sentiment_0shot.csv")

sentiment_map = {"negative": -1, "neutral": 0, "positive": 1}
comment_sentiment["sentiment_numeric"] = comment_sentiment["label"].map(sentiment_map)


df = (
    transcript_topics
    .merge(comment_topics, on="video_id", suffixes=("_transcript", "_comment"))
    .merge(comment_sentiment[["video_id", "sentiment_numeric"]], on="video_id")
)

print(f"Aligned videos: {len(df)}")


topic_cols_transcript = [c for c in df.columns if c.endswith("_transcript")]
topic_cols_comment = [c for c in df.columns if c.endswith("_comment")]

def topic_divergence(row):
    t = row[topic_cols_transcript].values
    c = row[topic_cols_comment].values

    if np.linalg.norm(t) == 0 or np.linalg.norm(c) == 0:
        return np.nan

    return cosine(t, c)

df["topic_divergence"] = df.apply(topic_divergence, axis=1)

df_cleaned = df.dropna(subset=["topic_divergence"])

rho, p = spearmanr(df_cleaned["topic_divergence"], df_cleaned["sentiment_numeric"])
print(f"Spearman ρ={rho:.3f}, p={p:.4g}")


plt.figure(figsize=(6, 5))
sns.regplot(
    data=df_cleaned,
    x="topic_divergence",
    y="sentiment_numeric",
    scatter_kws={"alpha": 0.5},
    lowess=True
)
plt.axhline(0, linestyle="--", color="gray")
plt.title("Topic Divergence vs Comment Sentiment")
plt.xlabel("Transcript–Comment Topic Divergence (Cosine)")
plt.ylabel("Comment Sentiment")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.histplot(df["topic_divergence"], bins=20, kde=True)
plt.xlabel("Transcript–Comment Topic Divergence")
plt.ylabel("Number of Videos")
plt.title("Distribution of Topic Divergence")

plt.subplot(1, 2, 2)
sns.histplot(df["sentiment_numeric"], bins=3, discrete=True)
plt.xlabel("Comment Sentiment")
plt.ylabel("Number of Videos")
plt.title("Distribution of Comment Sentiment")

plt.tight_layout()
plt.show()

bins = np.linspace(0, 1, 6)
labels = [f"{bins[i]:.1f}-{bins[i+1]:.1f}" for i in range(len(bins)-1)]

df["divergence_bin"] = pd.cut(df["topic_divergence"], bins=bins, labels=labels, include_lowest=True)

mean_sentiment_per_bin = df.groupby("divergence_bin")["sentiment_numeric"].mean().reset_index()

plt.figure(figsize=(8, 5))
sns.barplot(x="divergence_bin", y="sentiment_numeric", data=mean_sentiment_per_bin, palette="coolwarm")
plt.axhline(0, color="gray", linestyle="--")
plt.xlabel("Transcript–Comment Topic Divergence (binned)")
plt.ylabel("Mean Comment Sentiment")
plt.title("Mean Comment Sentiment vs Divergence")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()